# 02 — Explore the Catalogue

Opens the local STAC catalogue produced by `01_find_and_catalogue.ipynb` and explores its
contents: filtering events by time, spatial extent, and collections; inspecting matchup
metadata; and visualising product footprints and collocation regions.

**Prerequisites:** run `01_find_and_catalogue.ipynb` first so that `example/catalogue/` exists.

In [ ]:
import datetime as dt
import os

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from eomatch import MatchupCatalogue

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
CATALOGUE_DIR = os.path.join(NOTEBOOK_DIR, "catalogue")

cat = MatchupCatalogue.open(CATALOGUE_DIR)
print("Opened:", cat.path)

## All events

Retrieve every event from the catalogue with no filters applied.

In [ ]:
all_events = cat.get_events()
print(f"Total events: {len(all_events)}")

for event in all_events:
    n = len(event.matchup_set) if event.matchup_set is not None else 0
    print(f"  {event.stac_id}  —  {n} matchup(s)")

## Filter by time window

`get_events` accepts `start_time` / `stop_time` as an inclusive overlap window.
An event matches if its own time window intersects `[start_time, stop_time]`.

In [ ]:
events_may = cat.get_events(
    start_time=dt.datetime(2022, 5, 1),
    stop_time=dt.datetime(2022, 5, 31, 23, 59, 59),
)
print(f"Events in May 2022: {len(events_may)}")

events_june = cat.get_events(
    start_time=dt.datetime(2022, 6, 1),
)
print(f"Events from June 2022 onwards: {len(events_june)}")

## Filter by collections and bounding box

In [ ]:
# Restrict to a specific sensor pair
events_pair = cat.get_events(collections=["LANDSAT_C2L1", "S2_MSI_L1C"])
print(f"LANDSAT_C2L1 × S2_MSI_L1C events: {len(events_pair)}")

# Restrict to a bounding box: [lon_min, lat_min, lon_max, lat_max]
events_europe = cat.get_events(bbox=[-10.0, 35.0, 40.0, 72.0])
print(f"Events overlapping Europe bbox:    {len(events_europe)}")

## Inspect a matchup in detail

In [ ]:
if not all_events:
    raise RuntimeError("No events found — run 01_find_and_catalogue.ipynb first.")

event = all_events[0]
mu = event.matchup_set[0]

print("Event")
print(f"  ID:          {event.stac_id}")
print(f"  Platforms:   {event.platforms}")
print(f"  Collections: {event.collections}")
print(f"  Start:       {event.start_time}")
print(f"  Stop:        {event.stop_time}")
print()
print("Matchup")
print(f"  Collocation bounds: {mu.collocation_region.bounds}")
print(f"  Collocation area:   {mu.collocation_region.area:.4f} deg²")
print(f"  Time diff (s):      {mu.time_diff_abs:.1f}")
print()
print("Products")
for p in mu.products:
    print(f"  [{p.collection}]  {p.id}")
    print(f"    start={p.start_time}  stop={p.stop_time}")

## Visualise footprints and collocation region

In [ ]:
colours = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

fig, ax = plt.subplots(figsize=(10, 7), subplot_kw={"projection": ccrs.PlateCarree()})
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0")
ax.gridlines(draw_labels=True, linewidth=0.3, color="grey")

for i, p in enumerate(mu.products):
    geom = p.geometry
    colour = colours[i % len(colours)]
    x, y = geom.exterior.xy
    ax.fill(x, y, alpha=0.2, color=colour, transform=ccrs.PlateCarree())
    ax.plot(x, y, color=colour, linewidth=1.5,
            transform=ccrs.PlateCarree(), label=f"{p.collection} — {p.id[:20]}…")

# Collocation region
col_region = mu.collocation_region
x, y = col_region.exterior.xy
ax.fill(x, y, alpha=0.4, color="tab:green", transform=ccrs.PlateCarree())
ax.plot(x, y, color="tab:green", linewidth=2,
        transform=ccrs.PlateCarree(), label="Collocation region")

# Auto-zoom to the bounding box of all geometries
all_geoms = [p.geometry for p in mu.products] + [col_region]
minx = min(g.bounds[0] for g in all_geoms)
miny = min(g.bounds[1] for g in all_geoms)
maxx = max(g.bounds[2] for g in all_geoms)
maxy = max(g.bounds[3] for g in all_geoms)
pad = 2.0
ax.set_extent([minx - pad, maxx + pad, miny - pad, maxy + pad], crs=ccrs.PlateCarree())

ax.legend(loc="best", fontsize=8)
ax.set_title(f"Matchup footprints — {event.stac_id}")
plt.tight_layout()
plt.show()

## STAC item links

Show the raw `derived_from` / `related` links stored in the matchup STAC item — these are
relative filesystem paths that pystac resolves when the catalogue is opened locally.

In [ ]:
import json

# Find the matchup item JSON on disk
mu_collection_id = mu.stac_collection_id
mu_item_id = mu.stac_id

mu_json_path = os.path.join(
    CATALOGUE_DIR, mu_collection_id, mu_item_id, f"{mu_item_id}.json"
)

with open(mu_json_path) as fh:
    mu_json = json.load(fh)

print("Links stored in matchup item JSON:")
for link in mu_json["links"]:
    if link["rel"] in ("derived_from", "related", "root", "parent", "collection"):
        print(f"  rel={link['rel']:<15} href={link['href']}")